# 02 — Classification

This notebook evaluates two classifiers trained to predict **priority** and **department** from ticket text:

- **Baseline**: TF-IDF (unigrams + bigrams) + Logistic Regression — trained with `scripts/train_classifier.py`
- **BERT**: DistilBERT fine-tuned with weighted cross-entropy loss and early stopping — trained with `scripts/train_classifier_bert.py` on the Bocconi HPC cluster (GPU)

The purpose of classification in this project is not just to get good numbers.
It serves as evidence that **priority and department labels carry real textual signal** —
i.e., that tickets belonging to different subgroups are linguistically distinguishable.
This supports treating the labels as meaningful proxies when analyzing generation quality disparities.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
from src import config

CLF   = config.TABLES / 'classification'
FIGS  = config.FIGURES / 'classification'

## 1. Overall comparison

The table below summarises accuracy and macro-F1 on the test set for both models and both targets.

In [ ]:
rows = []
for model, folder in [('LogReg', ''), ('BERT', 'bert_')]:
    for target in ['priority', 'department']:
        path = CLF / f'{folder}{target}' / 'test_summary.json'
        s = json.loads(path.read_text())
        rows.append({'model': model, 'target': target,
                     'accuracy': s['accuracy'], 'f1_macro': s['f1_macro']})

comp = pd.DataFrame(rows)
comp

**Key observation:** BERT outperforms LogReg on both targets by a substantial margin
(~+10 points on priority, ~+15 points on department). Both models perform well above
random chance (33% for priority, 17% for department), confirming that the labels
are recoverable from ticket text.

The fact that even a linear TF-IDF model achieves ~65% on priority suggests
that subgroup differences are at least partially lexical — the same vocabulary
patterns that make a ticket classifiable may also influence how a generation model responds.

## 2. Priority classification

### 2a. Logistic Regression

In [ ]:
print((CLF / 'priority' / 'test_report.txt').read_text())

In [ ]:
display(Image(filename=str(FIGS / 'priority' / 'priority_test_confusion.png')))

### 2b. BERT (DistilBERT fine-tuned)

In [ ]:
print((CLF / 'bert_priority' / 'test_report.txt').read_text())

In [ ]:
display(Image(filename=str(FIGS / 'bert_priority' / 'test_confusion.png')))

**Observation:** BERT improves all three priority classes, including `low` which was the hardest
for LogReg (F1 0.59 → 0.72). The improvement on `low` is particularly relevant because
it is the minority class — weighted loss during fine-tuning helps compensate for imbalance.

## 3. Department classification

### 3a. Logistic Regression

In [ ]:
print((CLF / 'department' / 'test_report.txt').read_text())

In [ ]:
display(Image(filename=str(FIGS / 'department' / 'department_test_confusion.png')))

### 3b. BERT (DistilBERT fine-tuned)

In [ ]:
print((CLF / 'bert_department' / 'test_report.txt').read_text())

In [ ]:
display(Image(filename=str(FIGS / 'bert_department' / 'confusion_matrix.png')))

**Observation:** BERT improves dramatically on department (+15 macro-F1 points).
Billing and Payments stands out with F1 0.88 — likely because billing tickets
contain highly distinctive vocabulary (invoice, payment, charge, refund).
`Other` (the merged minority departments) achieves F1 0.73, reasonable given
it aggregates heterogeneous ticket types.

## Summary

- Both labels are **learnable from ticket text**, well above chance for both models.
- BERT outperforms LogReg on both targets, especially department (+15 F1 points).
- The synthetic nature of the data likely inflates performance — patterns may be more
  regular than in real-world tickets.
- **Implication for generation analysis**: since labels carry textual signal, subgroups
  defined by priority and department represent genuinely different kinds of input.
  Differences in generation quality across these subgroups are therefore interpretable
  as responses to different linguistic demands, not just arbitrary metadata splits.